In [1]:
import pandas as pd

In [2]:
demandas = pd.read_csv('demandas.csv', parse_dates=['data_inicio', 'data_fim'])

FIM_COLETA = demandas['data_fim'].max()   # fixado antes de qualquer filtro

funil = []

def registrar(nome, antes, depois):
    funil.append({'regra': nome, 'antes': antes, 'removidas': antes - depois, 'depois': depois})
    print(f"{nome}: {antes} → {depois} ({antes - depois} removidas)")

print(f"Base inicial: {len(demandas)} demandas | fim da coleta: {FIM_COLETA.date()}")

Base inicial: 383 demandas | fim da coleta: 2026-08-07


In [3]:
orcamentos = demandas[demandas['tag'] == 'Orçamento']
print(f"{len(orcamentos)} demandas de orçamento manual")
print(f"{orcamentos['esforco_horas'].sum():.2f}h gastas produzindo estimativas manualmente")
print(f"{orcamentos['esforco_horas'].sum() / demandas['esforco_horas'].sum() * 100:.1f}% do esforço total do período")

12 demandas de orçamento manual
12.44h gastas produzindo estimativas manualmente
1.1% do esforço total do período


In [4]:
antes = len(demandas)
demandas = demandas[demandas['tag'] != 'Orçamento'].reset_index(drop=True)
registrar('Fora de escopo: tag Orçamento (meta-trabalho, não execução)', antes, len(demandas))

Fora de escopo: tag Orçamento (meta-trabalho, não execução): 383 → 371 (12 removidas)


In [5]:
antes = len(demandas)
demandas = demandas[demandas['sistema'] != 'Sistema_001'].reset_index(drop=True)
registrar('Fora de escopo: produto interno (Sistema_001 — sem chamado formal)', antes, len(demandas))

Fora de escopo: produto interno (Sistema_001 — sem chamado formal): 371 → 351 (20 removidas)


In [6]:
antes = len(demandas)
demandas = demandas[demandas['esforco_horas'] >= 0.25].reset_index(drop=True)
registrar('Triviais: esforço < 0,25h (não passam por orçamentação)', antes, len(demandas))


Triviais: esforço < 0,25h (não passam por orçamentação): 351 → 326 (25 removidas)


In [7]:
antes = len(demandas)
corte = FIM_COLETA - pd.Timedelta(days=14)
demandas = demandas[demandas['data_fim'] <= corte].reset_index(drop=True)
registrar('Censura: demandas possivelmente em andamento (últimos 14 dias)', antes, len(demandas))

Censura: demandas possivelmente em andamento (últimos 14 dias): 326 → 313 (13 removidas)


In [8]:
antes = len(demandas)
demandas = demandas[demandas['chamado_mascarado'] != 'Chamado_142'].reset_index(drop=True)
registrar('Fora de escopo: projeto para uso interno da empresa (não orçado a cliente)', antes, len(demandas))

Fora de escopo: projeto para uso interno da empresa (não orçado a cliente): 313 → 312 (1 removidas)


In [9]:
pd.DataFrame(funil)

,regra,antes,removidas,depois
0,"Fora de escopo: tag Orçamento (meta-trabalho, ...",383,12,371
1,Fora de escopo: produto interno (Sistema_001 —...,371,20,351
2,"Triviais: esforço < 0,25h (não passam por orça...",351,25,326
3,Censura: demandas possivelmente em andamento (...,326,13,313
4,Fora de escopo: projeto para uso interno da em...,313,1,312


In [10]:
demandas.to_csv('demandas_filtradas.csv', index=False)
print(f"{len(demandas)} demandas salvas")

312 demandas salvas
